# Pre-EDA: Entendendo os Datasets
    Objetivo:
    Explorar e entender a estrutura inicial dos datasets (2022, 2023, 2024),
    identificando inconsistências, diferenças de schema e necessidades de limpeza.

## Importando as bibliotecas necessárias

In [6]:
import pandas as pd
import datetime

## Lendo o arquivo Excel

In [7]:
# Carregando os dados - notar que aqui as abas estão nomeadas como 
# "PEDE2022", "PEDE2023" e "PEDE2024"
arquivo =  "../data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

df_2022 = pd.read_excel(arquivo, sheet_name="PEDE2022")
df_2023 = pd.read_excel(arquivo, sheet_name="PEDE2023")
df_2024 = pd.read_excel(arquivo, sheet_name="PEDE2024")

## Colunas - Transformando e padronizando

In [4]:
# Cria funcao "limpar_colunas" para limpar e padronizar os nomes das colunas
def limpar_colunas(df):
    df.columns = (
        df.columns.str.strip()  # Remove espaços em branco
        .str.lower()  # Converte para minúsculas
        .str.replace(" ", "_")  # Substitui espaços por underscores
        .str.replace("(", "")  # Remove parênteses
        .str.replace(")", "")  # Remove parênteses
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8") # remove acentos        
    )
    return df

In [5]:
# aplica a função de limpeza em cada DataFrame
df_2022 = limpar_colunas(df_2022)
df_2023 = limpar_colunas(df_2023)
df_2024 = limpar_colunas(df_2024)

In [7]:
# identificando as colunas comuns entre os anos
colunas_comuns = set(df_2022.columns) & set(df_2023.columns) & set(df_2024.columns)

# exibindo as colunas que não são comuns
colunas_2022 = set(df_2022.columns)
colunas_2023 = set(df_2023.columns)
colunas_2024 = set(df_2024.columns)
colunas_unicas_2022 = colunas_2022 - colunas_comuns
colunas_unicas_2023 = colunas_2023 - colunas_comuns
colunas_unicas_2024 = colunas_2024 - colunas_comuns

A coluna 'destaque_ipv.1' do 'df_2023' foi originada a partir de uma coluna duplicada durante a transformação dos dados para o ano de 20203, e ela contém os mesmos valores que a coluna 'destaque_ipv', então podemos considerar apenas uma delas para a análise, e a outra pode ser descartada para evitar redundância.

A mesma situação ocorre para a coluna 'ativo/_inativo.1' do 'df_2024', que também foi criada a partir de uma coluna duplicada durante a transformação dos dados para o ano de 2024, e contém os mesmos valores que a coluna 'ativo/_inativo', então podemos considerar apenas uma delas para a análise, e a outra pode ser descartada para evitar redundância.

In [8]:
# removendo a coluna 'destaque_ipv.1' do df_2023
df_2023.drop(columns=['destaque_ipv.1'], inplace=True)
# removendo a coluna 'ativo/_inativo.1' do df_2024
df_2024.drop(columns=['ativo/_inativo.1'], inplace=True)

As colunas de pedras estão no padrão 'pedra_aa' e 'pedra_aaaa', onde 'aa' representa os dois últimos dígitos do ano correspondente, ou seja, 'pedra_23' para o ano de 2023 e 'aaaa' rempresenta os quatro dígitos do ano correspondente, ou seja, 'pedra_2023' para o ano de 2023. 

No df_2023, a coluna 'pedra_23' está vazia, enquanto a coluna 'pedra_2023' contém os dados corretos. No df_2024, no lugar da coluna 'pedra_24' temos 'pedra_2024', que contém os dados corretos. 

Optaremos por manter o padrão 'pedra_aa' para as colunas de pedras, ou seja, 'pedra_23' para o ano de 2023 e 'pedra_24' para o ano de 2024, e descartaremos as colunas 'pedra_2023' para evitar redundância e manter a consistência nos nomes das colunas.

Para isso, vamos remover as colunas 'pedra_23' do df_2023, e manter as colunas 'pedra_2023' e 'pedra_2024' respectivamente. Em seguida, vamos renomear as colunas 'pedra_2023' para 'pedra_23' e 'pedra_2024' para 'pedra_24' para manter o padrão de nomenclatura consistente.

In [9]:
# removendo a coluna 'pedra_23' do df_2023
df_2023.drop(columns=['pedra_23'], inplace=True)
# renomeando a coluna 'pedra_2023' para 'pedra_23' no df_2023
df_2023.rename(columns={'pedra_2023': 'pedra_23'}, inplace=True)
# renomeando a coluna 'pedra_2024' para 'pedra_24' no df_2024
df_2024.rename(columns={'pedra_2024': 'pedra_24'}, inplace=True)
# removendo a coluna 'inde_23' do df_2023
df_2023.drop(columns=['inde_23'], inplace=True)
# renomeando a coluna 'inde_2023' para 'inde_23' no df_2023
df_2023.rename(columns={'inde_2023': 'inde_23'}, inplace=True)
# removendo a coluna 'inde_24' do df_2024
df_2024.rename(columns={'inde_2024': 'inde_24'}, inplace=True)

In [10]:
# convertendo o nome das colunas do df_2022 para o mesmo nome dos outros anos
df_2022.rename(
    columns={
        "nome": "nome_anonimizado",
        "portug": "por",
        "defas": "defasagem",
        "ano_nasc": "data_de_nasc",
        "idade_22": "idade",
        "ingles": "ing",
        "matem": "mat",
    },
    inplace=True,
)

# Adicionando as colunas 'rec_av3' e 'rec_av4' para o df_2024
df_2024['rec_av3'] = pd.NA
df_2024['rec_av4'] = pd.NA

# Adicionando a coluna "ano_base" para cada DataFrame
df_2022['ano_base'] = 2022
df_2023['ano_base'] = 2023
df_2024['ano_base'] = 2024

In [11]:
# identificando as colunas comuns entre os anos
colunas_comuns = set(df_2022.columns) & set(df_2023.columns) & set(df_2024.columns)

# exibindo as colunas que não são comuns
colunas_2022 = set(df_2022.columns)
colunas_2023 = set(df_2023.columns)
colunas_2024 = set(df_2024.columns)
colunas_unicas_2022 = colunas_2022 - colunas_comuns
colunas_unicas_2023 = colunas_2023 - colunas_comuns
colunas_unicas_2024 = colunas_2024 - colunas_comuns

## Unindo os dataframes

In [12]:
# Unindo os DataFrames em um único DataFrame
df_completo = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

## Dados - Transformando e padronizando

a coluna 'ra' e 'nome_anonimizado' trazem basicamente o mesmo conteúdo, sendo RA-1 equivalente a Aluno-1, RA-2 equivalente a Aluno-2 e assim por diante. Dessa forma podemos remover a coluna 'nome_anonimizado' e manter apenas a coluna 'nome_anonimizado' para representar os alunos de forma anônima.

In [13]:
#removendo a coluna "nome_anonimizado"
df_completo.drop(columns=['nome_anonimizado'], inplace=True)

Considarando que a coluna 'ra' é composta por um prefixo 'RA-' seguido de um número, podemos extrair este prefixo e manter apenas o número para facilitar a análise e comparação dos dados. Para isso, podemos utilizar a função de string do pandas para remover o prefixo 'RA-' e manter apenas os números correspondentes aos alunos.

In [14]:
# removendo o prefixo 'RA-' da coluna 'ra'
df_completo['ra'] = df_completo['ra'].str.replace('RA-', '', regex=False)
# convertendo a coluna 'ra' para inteiro, tratando erros de conversão   
df_completo['ra'] = pd.to_numeric(df_completo['ra'], errors='coerce').astype('Int64')

In [15]:
# Substitui 'ALFA' da coluna 'fase' por '0' antes de tentar extrair o número
df_completo['fase'] = df_completo['fase'].astype('string') # evita erros caso haja valores numéricos misturados com strings
df_completo['fase'] = df_completo['fase'].replace('ALFA', '0')
df_completo['fase'] = df_completo['fase'].str.extract('(\d+)').astype('Int64')

In [16]:
# convertendo a coluna 'turma' para string
df_completo['turma'] = df_completo['turma'].astype('string')
#convertendo o valor '9' da coluna 'turma' em 'N/A'
df_completo['turma'] = df_completo['turma'].replace('9', 'a definir')
# convertendo a turma de 'ALFA C - G0/G1' para 'C' e fazendo o mesmo para as outras turmas
df_completo['turma'] = df_completo['turma'].str.replace('ALFA ', '', regex=False)
df_completo['turma'] = df_completo['turma'].str.replace(' - G0/G1', '', regex=False)
df_completo['turma'] = df_completo['turma'].str.replace(' - G2/G3', '', regex=False)
# agora converte os valores '1A', '1B', '1C', etc. para apenas 'A', 'B', 'C', etc.
df_completo['turma'] = df_completo['turma'].str.replace(r'^\d+', '', regex=True)

In [17]:
# padronizando a coluna 'genero' para ter apenas 'masculino' e 'feminino'
df_completo['genero'] = df_completo['genero'].str.lower().str.strip()
df_completo['genero'] = df_completo['genero'].replace({'menino': 'masculino', 'menina': 'feminino'})

In [18]:
# padronizando as colunas das pedras de 'pedra_20' a 'pedra_24'
# remove acentos e normaliza os nomes das colunas de pedra
def padronizar_pedra(coluna):
    coluna = coluna.str.lower().str.strip()
    coluna = coluna.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    return coluna

for i in range(20, 25):
    df_completo[f'pedra_{i}'] = padronizar_pedra(df_completo[f'pedra_{i}'])
# converte converte o valor 'incluir' em nulo
for i in range(20, 25):
    df_completo[f'pedra_{i}'] = df_completo[f'pedra_{i}'].replace('incluir', pd.NA)

In [20]:
# convertendo a data de narcimento que contem apenas o ano para o formato "01/01/AAAA" para manter a consistência com os outros anos
df_completo['data_de_nasc'] = df_completo['data_de_nasc'].apply(lambda x: f"01/01/{x}" if isinstance(x, (int, float)) else x)   
# convertendo a coluna 'data_de_nasc' do formato timestamp para datetime no formato "DD/MM/AAAA", tratando erros com 'coerce' para evitar problemas de conversão
df_completo['data_de_nasc'] = pd.to_datetime(df_completo['data_de_nasc'], errors='coerce').dt.strftime('%d/%m/%Y')

In [21]:
def limpar_idade(valor):
    # Se for um objeto de data, pegamos apenas o dia
    if isinstance(valor, datetime.datetime):
        return valor.day
    # Se for um número (ou string de número), convertemos para inteiro
    try:
        return int(float(valor))
    except:
        return None # Caso haja algum valor bizarro que não seja data nem número

# Aplicando a função na coluna
df_completo['idade'] = df_completo['idade'].apply(limpar_idade).astype('Int64')

In [22]:
# transforma os nulos de 'instituicao_de_ensino' em 'outros'
df_completo['instituicao_de_ensino'] = df_completo['instituicao_de_ensino'].fillna('outros')
# transforma os nulos de 'escola' em 'outros'
df_completo['escola'] = df_completo['escola'].fillna('outros')

In [23]:
# converte a coluna 'ano_ingresso' em inteiro, tratando erros com 'coerce' para evitar problemas de conversão
df_completo['ano_ingresso'] = pd.to_numeric(df_completo['ano_ingresso'], errors='coerce').astype('Int64')
df_completo['cg'] = pd.to_numeric(df_completo['cg'], errors='coerce').astype('Int64')
df_completo['cf'] = pd.to_numeric(df_completo['cf'], errors='coerce').astype('Int64')
df_completo['ct'] = pd.to_numeric(df_completo['ct'], errors='coerce').astype('Int64')
df_completo['no_av'] = pd.to_numeric(df_completo['no_av'], errors='coerce').astype('Int64')
df_completo['defasagem'] = pd.to_numeric(df_completo['defasagem'], errors='coerce').astype('Int64')
df_completo['ano_base'] = pd.to_numeric(df_completo['ano_base'], errors='coerce').astype('Int64')

# converte a coluna 'inde_22' em float, tratando erros com 'coerce' para evitar problemas de conversão
df_completo['inde_22'] = pd.to_numeric(df_completo['inde_22'], errors='coerce').astype('Float64')
df_completo['inde_23'] = pd.to_numeric(df_completo['inde_23'], errors='coerce').astype('Float64')
df_completo['iaa'] = pd.to_numeric(df_completo['iaa'], errors='coerce').astype('Float64')
df_completo['ieg'] = pd.to_numeric(df_completo['ieg'], errors='coerce').astype('Float64')
df_completo['ips'] = pd.to_numeric(df_completo['ips'], errors='coerce').astype('Float64')
df_completo['ida'] = pd.to_numeric(df_completo['ida'], errors='coerce').astype('Float64')
df_completo['mat'] = pd.to_numeric(df_completo['mat'], errors='coerce').astype('Float64')
df_completo['por'] = pd.to_numeric(df_completo['por'], errors='coerce').astype('Float64')
df_completo['ing'] = pd.to_numeric(df_completo['ing'], errors='coerce').astype('Float64')
df_completo['ipv'] = pd.to_numeric(df_completo['ipv'], errors='coerce').astype('Float64')
df_completo['ian'] = pd.to_numeric(df_completo['ian'], errors='coerce').astype('Float64')
df_completo['ipp'] = pd.to_numeric(df_completo['ipp'], errors='coerce').astype('Float64')

# converte a coluna 'rec_av1' em string, tratando erros com 'coerce' para evitar problemas de conversão
df_completo['rec_av1'] = df_completo['rec_av1'].astype('string')
df_completo['rec_av2'] = df_completo['rec_av2'].astype('string')
df_completo['rec_av3'] = df_completo['rec_av3'].astype('string')
df_completo['rec_av4'] = df_completo['rec_av4'].astype('string')
df_completo['rec_psicologia'] = df_completo['rec_psicologia'].astype('string')
df_completo['indicado'] = df_completo['indicado'].astype('string')
df_completo['atingiu_pv'] = df_completo['atingiu_pv'].astype('string')
df_completo['destaque_ieg'] = df_completo['destaque_ieg'].astype('string')
df_completo['destaque_ida'] = df_completo['destaque_ida'].astype('string')
df_completo['destaque_ipv'] = df_completo['destaque_ipv'].astype('string')

In [24]:
# converte o valor 'INCLUIR' da coluna 'inde_24' em nulo
df_completo['inde_24'] = df_completo['inde_24'].replace('INCLUIR', pd.NA)
# converte a coluna 'inde_24' em float, tratando erros com 'coerce' para evitar problemas de conversão
df_completo['inde_24'] = pd.to_numeric(df_completo['inde_24'], errors='coerce').astype('Float64')

Apesar das colunas 'avaliador6', 'avaliador5', 'rec_av4' trazerem mais de 90% dos dados nulos, elas ainda contêm algumas informações relevantes para a análise, e por isso optamos por mantê-las no dataframe. Observa-se que quanto melhor a nota dos alunos, menor a quantidade de dados nulos nessas colunas, o que pode indicar que os alunos com melhor desempenho tendem a ter menos avaliações registradas. Portanto, mesmo com a presença de dados nulos, essas colunas ainda podem fornecer insights valiosos sobre o desempenho dos alunos e suas avaliações.

## Agrupando as colunas de acordo com o tipo de informação

Para organizar as 51 colunas de forma lógica, o ideal é agrupá-las por domínio de informação. Isso facilita tanto a criação de gráficos no Streamlit quanto a seleção de features para o seu modelo de Machine Learning.
1. Identificação e Demografia: Dados básicos do aluno.
2. Contexto Escolar e Status: Onde estuda e situação atual no programa.
3. Indicadores de Performance (Série "I"): Os índices fundamentais (IDA, IEG, IAA, etc.) que o desafio pede para analisar.
4. Histórico e Evolução Global: Resultados do INDE e a classificação por "Pedras" ao longo dos anos.
5. Componentes de Notas: Notas por disciplina e conceitos gerais.
6. Avaliações e Feedbacks: Colunas qualitativas (comentários de avaliadores e destaques).

In [25]:
# Definindo as listas de colunas por grupo
cols_identificacao = ['ra', 'genero', 'data_de_nasc', 'idade']

cols_contexto = ['escola', 'instituicao_de_ensino', 'ano_ingresso', 'fase', 'turma', 'fase_ideal', 'ativo/_inativo' , 'defasagem']

# Os indicadores principais que são o core do desafio
cols_indicadores_chave = ['ian', 'ida', 'ieg', 'iaa', 'ips', 'ipp', 'ipv', 'atingiu_pv']

# Disciplinas e sub-conceitos
cols_notas_disciplinas = ['mat', 'por', 'ing', 'cg', 'cf', 'ct']

# Evolução temporal do índice global e classificação
cols_historico_global = [
    'pedra_20', 'pedra_21', 
    'pedra_22', 'inde_22', 
    'pedra_23', 'inde_23', 
    'pedra_24', 'inde_24'
]

# Dados de quem avaliou e textos de recomendação/destaque
cols_avaliacoes_feedback = [
    'no_av', 'avaliador1', 'rec_av1', 'avaliador2', 'rec_av2', 
    'avaliador3', 'rec_av3', 'avaliador4', 'rec_av4', 'avaliador5', 'avaliador6', 
    'rec_psicologia', 'indicado', 'destaque_ieg', 'destaque_ida', 'destaque_ipv'
]

cols_metadados = ['ano_base']

# Criando a nova ordem completa
nova_ordem = (cols_identificacao + cols_contexto + cols_indicadores_chave + 
              cols_notas_disciplinas + cols_historico_global + 
              cols_avaliacoes_feedback + cols_metadados)

# Reordenando o DataFrame
df_completo = df_completo[nova_ordem]

## Gerando o arquivo ajustado

In [ ]:
df_completo.to_csv("../data/interim/pede_interim.csv", sep=";", index=False)